In [1]:
# 05c_soft_conflict_diagnosis.ipynb
# =============================================================================
# Notebook 05c — Diagnose why Guardrail_soft INCREASES sodium-carb conflict
#
# Tests three hypotheses for the 31.2% -> 42.6% conflict increase under
# permitted_range injection (before hard clip):
#
#   H1: the guardrail range itself permits "sodium down + carb/sugar up".
#   H2: guardrail forces sodium down (ceiling 0.9c), so more CFs ENTER the
#       conflict-eligible set (sodium decreased) -> larger denominator.
#   H3: build_permitted widens the carb UPPER bound via data-quantile
#       (max(hi, data_hi)), overriding the hard-rule ceiling (carb <= current),
#       so carb-up becomes reachable.
#
# Diagnostic outputs per condition:
#   - n_sodium_decreased : how many CFs actually cut sodium (conflict-eligible)
#   - conflict_rate_among_eligible : conflicts / sodium-decreased  (removes H2 artefact)
#   - n_carb_ceiling_violated : CFs where carb exceeds patient baseline
#   - n_permitted_allows_conflict : cases whose injected range upper bounds
#       let carb/sugar rise above baseline while sodium ceiling < baseline
# =============================================================================

# %%
import json, joblib, warnings
import numpy as np, pandas as pd
import dice_ml
warnings.filterwarnings('ignore')

CHANGE_TOL = 1.0

agent_config = joblib.load('../results/tables/agent_config.pkl')
df_final     = joblib.load('../results/tables/df_final.pkl')
X_FEATURES      = agent_config['X_features']
VARY_FEATURES   = agent_config['vary_features']
TARGET_COL      = agent_config['target_col']
AGEGROUP_CONFIG = agent_config['agegroup_config']
models = {g: joblib.load(f'../results/tables/model_{g}.pkl') for g in AGEGROUP_CONFIG}
with open('../results/tables/guardrail_ranges_v2.json', encoding='utf-8') as f:
    GR = json.load(f)

def build_permitted(ranges, df_ref, features):
    sr = {}
    for f in features:
        pair = ranges.get(f)
        if pair is None:
            continue
        lo, hi = float(pair[0]), float(pair[1])
        dlo = float(df_ref[f].quantile(0.05)); dhi = float(df_ref[f].quantile(0.95))
        sr[f] = [round(max(min(lo, dlo), 0.0), 4), round(max(hi, dhi), 4)]
    return sr

def build_permitted_strict(ranges, features):
    """Same ranges but WITHOUT data-quantile widening (tests H3)."""
    sr = {}
    for f in features:
        pair = ranges.get(f)
        if pair is None:
            continue
        sr[f] = [round(float(pair[0]),4), round(float(pair[1]),4)]
    return sr

def gen_cfs(exp, query, permitted=None, n=4):
    kw = dict(total_CFs=n, desired_class=0, features_to_vary=VARY_FEATURES,
              proximity_weight=0.2, sparsity_weight=0.1)
    if permitted:
        kw['permitted_range'] = permitted
    try:
        cf = exp.generate_counterfactuals(query, **kw)
        return cf.cf_examples_list[0].final_cfs_df.to_dict('records') if cf else []
    except Exception:
        return []

def sodium_decreased(cf, orig, tol=CHANGE_TOL):
    c = float(orig.get('Sodium_mg',0)); v = float(cf.get('Sodium_mg',c))
    return c > 0 and (c - v)/c*100 > tol

def carb_or_sugar_up(cf, orig, tol=CHANGE_TOL):
    c_c=float(orig.get('Carb_g',0)); v_c=float(cf.get('Carb_g',c_c))
    c_s=float(orig.get('Sugar_g',0)); v_s=float(cf.get('Sugar_g',c_s))
    up_c = (v_c-c_c)/c_c*100 > tol if c_c>0 else False
    up_s = (v_s-c_s)/c_s*100 > tol if c_s>0 else False
    return up_c or up_s

def carb_above_baseline(cf, orig, tol=CHANGE_TOL):
    c=float(orig.get('Carb_g',0)); v=float(cf.get('Carb_g',c))
    return (v-c)/c*100 > tol if c>0 else False

# %%
# H1 & permitted-analysis: does the injected range allow carb/sugar > baseline
# while sodium ceiling < baseline?
print("=== H1: does injected permitted_range allow conflict? ===")
h1_rows = []
for case_key, g in GR.items():
    orig = g['patient_profile']
    fr = g['final_ranges']
    na_ceiling = fr['Sodium_mg'][1]
    na_base = float(orig['Sodium_mg'])
    carb_ceiling = fr['Carb_g'][1]
    carb_base = float(orig['Carb_g'])
    sugar_ceiling = fr['Sugar_g'][1]
    sugar_base = float(orig['Sugar_g'])
    # after data-quantile widening (what was actually injected)
    grp=g['group']; cfg=AGEGROUP_CONFIG[grp]
    age=0.0 if cfg['age_min']<60 else 1.0
    dref=df_final[(df_final['AgeGroup']==age)&(df_final['Sex']==cfg['sex_code'])]
    carb_ceiling_widened = max(carb_ceiling, float(dref['Carb_g'].quantile(0.95)))
    sugar_ceiling_widened = max(sugar_ceiling, float(dref['Sugar_g'].quantile(0.95)))
    h1_rows.append({
        'CaseKey':case_key,
        'Na_ceiling<base': na_ceiling < na_base,
        'Carb_ceiling(final)>base': carb_ceiling > carb_base*1.01,
        'Carb_ceiling(injected)>base': carb_ceiling_widened > carb_base*1.01,
        'Sugar_ceiling(injected)>base': sugar_ceiling_widened > sugar_base*1.01,
    })
h1 = pd.DataFrame(h1_rows)
print(h1.to_string(index=False))
print(f"\n  cases where injected carb ceiling exceeds baseline: "
      f"{int(h1['Carb_ceiling(injected)>base'].sum())}/12")
print(f"  cases where injected sugar ceiling exceeds baseline: "
      f"{int(h1['Sugar_ceiling(injected)>base'].sum())}/12")

# %%
# H2 & H3: rerun soft with widened vs strict permitted, measure eligible set
print("\n=== H2/H3: eligible-set and strict-range diagnosis ===")
diag = []
for case_key, g in GR.items():
    grp=g['group']; mdl=models.get(grp)
    if mdl is None: continue
    orig=g['patient_profile']; cfg=AGEGROUP_CONFIG[grp]
    age=0.0 if cfg['age_min']<60 else 1.0
    dref=df_final[(df_final['AgeGroup']==age)&(df_final['Sex']==cfg['sex_code'])].copy()
    query=pd.DataFrame([orig])[X_FEATURES]
    d=dice_ml.Data(dataframe=df_final.copy().astype(float)[X_FEATURES+[TARGET_COL]],
                   continuous_features=X_FEATURES, outcome_name=TARGET_COL)
    m=dice_ml.Model(model=mdl, backend='sklearn')
    exp=dice_ml.Dice(d,m,method='genetic')

    perm_wide   = build_permitted(g['final_ranges'], dref, X_FEATURES)
    perm_strict = build_permitted_strict(g['final_ranges'], X_FEATURES)

    for label, perm in [('pure',None),('soft_wide',perm_wide),('soft_strict',perm_strict)]:
        cfs = gen_cfs(exp, query, perm)
        n=len(cfs)
        n_elig = sum(sodium_decreased(cf,orig) for cf in cfs)
        n_conf = sum(sodium_decreased(cf,orig) and carb_or_sugar_up(cf,orig) for cf in cfs)
        n_carb_over = sum(carb_above_baseline(cf,orig) for cf in cfs)
        diag.append({'CaseKey':case_key,'Cond':label,'n_CF':n,
                     'n_Na_decreased':n_elig,
                     'n_conflict':n_conf,
                     'conflict_among_eligible': round(n_conf/n_elig,2) if n_elig else np.nan,
                     'n_carb_above_baseline':n_carb_over})
    print(f"  [{case_key}] done")

dd = pd.DataFrame(diag)
dd.to_csv('../results/tables/soft_conflict_diagnosis.csv', index=False, encoding='utf-8-sig')

# %%
print("\n=== Aggregate by condition ===")
agg = dd.groupby('Cond').agg(
    n_CF=('n_CF','sum'),
    n_Na_decreased=('n_Na_decreased','sum'),
    n_conflict=('n_conflict','sum'),
    n_carb_above_baseline=('n_carb_above_baseline','sum'),
).reindex(['pure','soft_wide','soft_strict'])
agg['conflict_rate_all_%'] = (agg['n_conflict']/agg['n_CF']*100).round(1)
agg['eligible_%'] = (agg['n_Na_decreased']/agg['n_CF']*100).round(1)
agg['conflict_among_eligible_%'] = (agg['n_conflict']/agg['n_Na_decreased']*100).round(1)
print(agg.to_string())

print("\nInterpretation guide:")
print("  - If eligible_% jumps pure->soft: H2 (guardrail forces sodium down,")
print("    enlarging the conflict-eligible set) explains part of the increase.")
print("  - If conflict_among_eligible_% is similar pure vs soft: the per-eligible")
print("    conflict propensity did NOT worsen; the raw-rate rise is a denominator effect.")
print("  - If soft_strict has far fewer carb_above_baseline than soft_wide: H3")
print("    (data-quantile widening overrode the carb ceiling) is a real driver,")
print("    and is FIXABLE by not widening ceilings above baseline.")

=== H1: does injected permitted_range allow conflict? ===
                CaseKey  Na_ceiling<base  Carb_ceiling(final)>base  Carb_ceiling(injected)>base  Sugar_ceiling(injected)>base
  MiddleAged_Male_case1             True                     False                         True                          True
  MiddleAged_Male_case2             True                     False                         True                          True
  MiddleAged_Male_case3             True                     False                         True                          True
MiddleAged_Female_case1             True                     False                         True                          True
MiddleAged_Female_case2             True                     False                         True                          True
MiddleAged_Female_case3             True                     False                         True                          True
       Older_Male_case1             True                    

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.23it/s]


  [MiddleAged_Male_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.86it/s]


  [MiddleAged_Male_case2] done


...

  [Older_Female_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.02s/it]


  [Older_Female_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.89s/it]

  [Older_Female_case3] done

=== Aggregate by condition ===
             n_CF  n_Na_decreased  n_conflict  n_carb_above_baseline  conflict_rate_all_%  eligible_%  conflict_among_eligible_%
Cond                                                                                                                            
pure           48              23          17                     20                 35.4        47.9                       73.9
soft_wide      47              28          22                     22                 46.8        59.6                       78.6
soft_strict    38              38           0                      0                  0.0       100.0                        0.0

Interpretation guide:
  - If eligible_% jumps pure->soft: H2 (guardrail forces sodium down,
    enlarging the conflict-eligible set) explains part of the increase.
  - If conflict_among_eligible_% is similar pure vs soft: the per-eligible
    conflict propensity did NOT worsen; the raw-rate ri